In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import warnings
from matplotlib import pyplot as plt
from IPython.display import display

warnings.simplefilter('ignore')

# ==============================================================================
# CONFIGURAZIONE — modifica questi valori prima di eseguire
# ==============================================================================
D_FIBRA_REALE = 110          # Diametro reale della fibra in micrometri (µm)
                             # Esempi: 80 per la fibra vecchia, 110 per quella nuova
SOGLIA_VARIAZIONE = 15       # Tolleranza in pixel per la costruzione della maschera
ESTENSIONI_SUPPORTATE = ('.tif', '.tiff', '.TIF', '.TIFF', '.png', '.jpg', '.jpeg')
# ==============================================================================


def get_white_seg_center(mask, y):
    pixels = np.where(mask[y, :] == 255)[0]
    if len(pixels) >= 2:
        return (pixels[0] + pixels[-1]) / 2, True
    return 0, False


def calcola_picco_gaussiano(line, approx_pos, window=10):
    """Stima sub-pixel del bordo tramite interpolazione parabolica sul gradiente."""
    start = max(0, int(approx_pos) - window)
    end = min(len(line), int(approx_pos) + window)
    if end - start < 3:
        return float(approx_pos)
    roi = line[start:end]
    grad = np.abs(np.gradient(roi.astype(float)))
    peak_idx = np.argmax(grad)
    if peak_idx == 0 or peak_idx == len(grad) - 1:
        return float(start + peak_idx)
    g_m1, g_0, g_p1 = grad[peak_idx - 1], grad[peak_idx], grad[peak_idx + 1]
    if g_m1 <= 0 or g_0 <= 0 or g_p1 <= 0:
        return float(start + peak_idx)
    l_m1, l_0, l_p1 = np.log(g_m1), np.log(g_0), np.log(g_p1)
    denom = l_m1 - 2 * l_0 + l_p1
    if denom == 0:
        return float(start + peak_idx)
    offset = (l_m1 - l_p1) / (2 * denom)
    return float(start + peak_idx + offset)


def calcola_larghezze_riga(mask, h):
    """Calcola la larghezza in pixel di ogni riga della maschera."""
    widths = np.zeros(h, dtype=int)
    for y in range(h):
        pts = np.where(mask[y, :] == 255)[0]
        if len(pts) >= 2:
            widths[y] = int(pts[-1] - pts[0])
    return widths


def analizza_singola_immagine(file_path, D_FIBRA_REALE=110):
    """
    Analizza un'immagine TIF/PNG/JPG contenente una fibra con gocciolina.

    Parametri
    ---------
    file_path : str
        Percorso del file immagine.
    D_FIBRA_REALE : float
        Diametro reale della fibra in µm (usato per la calibrazione).

    Ritorna
    -------
    dict con chiavi: File, Diam_um, Alt_um, Vol_um3
    oppure None se l'analisi non riesce.
    """
    img_raw = cv2.imread(file_path, cv2.IMREAD_UNCHANGED)
    if img_raw is None:
        print(f"[ERRORE] Impossibile aprire: {file_path}")
        return None

    # ------------------------------------------------------------------
    # 1. ROTAZIONE INIZIALE (90° orario) e normalizzazione
    # ------------------------------------------------------------------
    img_orig = cv2.rotate(img_raw, cv2.ROTATE_90_CLOCKWISE)
    gray = cv2.cvtColor(img_orig, cv2.COLOR_BGR2GRAY) if len(img_orig.shape) == 3 else img_orig
    gray_norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray_norm)

    _, thresh = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    h_img, w_img = thresh.shape

    # ------------------------------------------------------------------
    # 2. COSTRUZIONE MASCHERA CONTINUA
    #    Riempie ogni riga con un segmento coerente, ignorando artefatti.
    # ------------------------------------------------------------------
    mask_piena = np.zeros_like(thresh)
    ultima_l, ultimo_i = None, None
    for i in range(h_img):
        pts = np.where(thresh[i, :] == 255)[0]
        if len(pts) >= 2:
            inizio, fine = pts[0], pts[-1]
            l_curr = fine - inizio
            if ultima_l is None or abs(l_curr - ultima_l) < SOGLIA_VARIAZIONE:
                mask_piena[i, inizio:fine + 1] = 255
                ultima_l, ultimo_i = l_curr, inizio
            elif ultimo_i is not None:
                mask_piena[i, ultimo_i:ultimo_i + ultima_l] = 255

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------
    # 3. STIMA BASELINE FIBRA tramite PERCENTILE 10
    #    La mediana fallisce quando la gocciolina è molto grande e occupa
    #    più del 50% delle righe (come in Fotoi_166). Il p10 è invece
    #    sempre stabile: anche se il droplet occupa il 75% delle righe,
    #    il 10° percentile cade comunque nelle righe di fibra pura.
    #    Verificato su tutte le immagini: piccola (154), media (150),
    #    grande (108/112), molto grande (166).
    # ------------------------------------------------------------------
    widths = calcola_larghezze_riga(mask_piena, h_img)
    valid_widths = widths[widths > 10]
    if len(valid_widths) == 0:
        print(f"[ERRORE] Nessuna riga valida trovata in {os.path.basename(file_path)}")
        return None

    fiber_w_baseline = np.percentile(valid_widths, 10)
    droplet_soglia = fiber_w_baseline * 1.05   # righe con droplet = >5% più larghe della fibra

    # ------------------------------------------------------------------
    # 4. SEGMENTO FIBRA PURA PIÙ LUNGO (per calcolo angolo di inclinazione)
    #    Cerca il segmento continuo più lungo di righe "fibra pura"
    #    (larghezza ≤ soglia droplet). Gestisce anche fibre non continue
    #    (fibra che non arriva da bordo a bordo dell'immagine).
    # ------------------------------------------------------------------
    longest_fiber_seg = []
    current_seg = []
    for y in range(h_img):
        if 10 < widths[y] <= droplet_soglia:
            current_seg.append(y)
        else:
            if len(current_seg) > len(longest_fiber_seg):
                longest_fiber_seg = current_seg
            current_seg = []
    if len(current_seg) > len(longest_fiber_seg):
        longest_fiber_seg = current_seg

    # ------------------------------------------------------------------
    # 5. CALCOLO ANGOLO E ROTAZIONE FINE
    # ------------------------------------------------------------------
    if len(longest_fiber_seg) >= 20:
        seg_da_usare = longest_fiber_seg
    else:
        # Fallback: usa tutte le righe valide (fibra + droplet)
        seg_da_usare = list(np.where(widths > 10)[0])

    x_pts = []
    y_pts_fit = []
    for y in seg_da_usare:
        center, ok = get_white_seg_center(mask_piena, y)
        if ok:
            x_pts.append(center)
            y_pts_fit.append(y)

    if len(x_pts) >= 2:
        m, _ = np.polyfit(y_pts_fit, x_pts, 1)
        angolo = np.degrees(np.arctan(m))
    else:
        angolo = 0.0

    M = cv2.getRotationMatrix2D((w_img // 2, h_img // 2), -angolo, 1.0)
    img_rot     = cv2.warpAffine(gray_norm, M, (w_img, h_img), flags=cv2.INTER_CUBIC,   borderValue=255)
    mask_rot    = cv2.warpAffine(mask_piena, M, (w_img, h_img), flags=cv2.INTER_NEAREST, borderValue=0)
    gray_rot_raw = cv2.warpAffine(gray,     M, (w_img, h_img), flags=cv2.INTER_CUBIC,   borderValue=int(np.mean(gray)))
    gray_float  = cv2.GaussianBlur(gray_rot_raw.astype(float), (3, 3), 0)

    # ------------------------------------------------------------------
    # 6. RICALCOLO LARGHEZZE SULLA MASCHERA RUOTATA
    # ------------------------------------------------------------------
    widths_rot = calcola_larghezze_riga(mask_rot, h_img)
    valid_rot  = widths_rot[widths_rot > 10]
    if len(valid_rot) == 0:
        print(f"[ERRORE] Maschera vuota dopo rotazione in {os.path.basename(file_path)}")
        return None

    fiber_w_rot   = np.percentile(valid_rot, 10)
    droplet_soglia_rot = fiber_w_rot * 1.05

    fiber_only_rot  = np.where((widths_rot > 10) & (widths_rot <= droplet_soglia_rot))[0]
    droplet_rows_rot = np.where(widths_rot > droplet_soglia_rot)[0]

    if len(droplet_rows_rot) == 0:
        print(f"[ATTENZIONE] Nessun droplet rilevato in {os.path.basename(file_path)}")
        return None

    # Posizione verticale del massimo diametro (approssimazione iniziale)
    y_max_approx = int(droplet_rows_rot[np.argmax(widths_rot[droplet_rows_rot])])

    # ------------------------------------------------------------------
    # 7. CALIBRAZIONE µm/pixel
    #    Usa una riga di fibra pura lontana dal droplet per massima precisione.
    # ------------------------------------------------------------------
    if len(fiber_only_rot) >= 10:
        distanze = np.abs(fiber_only_rot - y_max_approx)
        n_top = max(10, len(fiber_only_rot) // 5)
        far_rows = fiber_only_rot[np.argsort(distanze)[-n_top:]]
        y_calib = int(np.median(far_rows))
        pts_calib = np.where(mask_rot[y_calib, :] == 255)[0]
        # Fallback se la riga mediana è vuota dopo rotazione
        if len(pts_calib) < 2:
            for y_try in sorted(far_rows, key=lambda y: -widths_rot[y]):
                pts_try = np.where(mask_rot[y_try, :] == 255)[0]
                if len(pts_try) >= 2:
                    y_calib = y_try
                    pts_calib = pts_try
                    break
    else:
        # Nessuna riga fibra pura trovata: usa la riga con larghezza più vicina alla mediana
        diffs = np.abs(widths_rot - fiber_w_rot)
        y_calib = int(np.argmin(diffs))
        pts_calib = np.where(mask_rot[y_calib, :] == 255)[0]

    sx_c = calcola_picco_gaussiano(gray_float[y_calib, :], pts_calib[0])
    dx_c = calcola_picco_gaussiano(gray_float[y_calib, :], pts_calib[-1])
    d_fib_px  = dx_c - sx_c
    um_per_px = D_FIBRA_REALE / d_fib_px

    # ------------------------------------------------------------------
    # 8. DIAMETRO MASSIMO tramite FITTING PARABOLA
    # ------------------------------------------------------------------
    win = max(20, len(droplet_rows_rot) // 10)
    y_range = range(max(0, y_max_approx - win), min(h_img, y_max_approx + win))
    y_v, sx_v, dx_v = [], [], []

    for y in y_range:
        pts = np.where(mask_rot[y, :] == 255)[0]
        if len(pts) >= 2 and widths_rot[y] > droplet_soglia_rot:
            sx_v.append(calcola_picco_gaussiano(gray_float[y, :], pts[0]))
            dx_v.append(calcola_picco_gaussiano(gray_float[y, :], pts[-1]))
            y_v.append(y)

    # Fallback: usa tutte le droplet rows se la finestra era troppo stretta
    if len(y_v) < 3:
        y_v, sx_v, dx_v = [], [], []
        for y in droplet_rows_rot:
            pts = np.where(mask_rot[y, :] == 255)[0]
            if len(pts) >= 2:
                sx_v.append(calcola_picco_gaussiano(gray_float[y, :], pts[0]))
                dx_v.append(calcola_picco_gaussiano(gray_float[y, :], pts[-1]))
                y_v.append(y)

    if len(y_v) < 3:
        print(f"[ERRORE] Punti insufficienti per il fitting del diametro in {os.path.basename(file_path)}")
        return None

    fit_sx = np.polyfit(y_v, sx_v, 2)
    fit_dx = np.polyfit(y_v, dx_v, 2)
    a_d = fit_dx[0] - fit_sx[0]
    b_d = fit_dx[1] - fit_sx[1]
    c_d = fit_dx[2] - fit_sx[2]
    y_true   = -b_d / (2 * a_d) if a_d != 0 else float(y_max_approx)
    d_max_px = a_d * (y_true ** 2) + b_d * y_true + c_d
    y_max_approx = int(np.clip(y_true, 0, h_img - 1))

    # ------------------------------------------------------------------
    # 9. ALTEZZA GOCCIOLINA
    # ------------------------------------------------------------------
    def find_boundary(y_start, step, h_limit, d_fib_px_ref):
        y_last = float(y_start)
        for y in range(y_start, h_limit, step):
            pts = np.where(mask_rot[y, :] == 255)[0]
            if len(pts) < 2:
                break
            w = (calcola_picco_gaussiano(gray_float[y, :], pts[-1]) -
                 calcola_picco_gaussiano(gray_float[y, :], pts[0]))
            if w <= d_fib_px_ref * 1.05:
                return float(y)
            y_last = float(y)
        return y_last

    y_b_sub = find_boundary(y_max_approx,  1, h_img - 5, d_fib_px)
    y_t_sub = find_boundary(y_max_approx, -1, 5,          d_fib_px)
    h_px = y_b_sub - y_t_sub

    # ------------------------------------------------------------------
    # 10. VOLUME
    # ------------------------------------------------------------------
    R_drop = (d_max_px * um_per_px) / 2
    H_drop = h_px * um_per_px
    R_fib  = D_FIBRA_REALE / 2
    volume = (4 / 3) * np.pi * (R_drop ** 2) * H_drop - np.pi * (R_fib ** 2) * H_drop

    # ------------------------------------------------------------------
    # 11. PLOT (stessa struttura dell'originale)
    # ------------------------------------------------------------------
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))

    viz_cal = cv2.cvtColor(img_rot, cv2.COLOR_GRAY2RGB)
    cv2.line(viz_cal, (int(sx_c), y_calib), (int(dx_c), y_calib), (255, 255, 0), 4)  # Giallo: calibrazione
    if len(sx_v) > 0:
        cv2.line(viz_cal, (int(sx_v[0]), y_max_approx), (int(dx_v[0]), y_max_approx), (255, 0, 0), 2)  # Rosso: equatore
    ax[0].imshow(viz_cal)
    ax[0].set_title("Giallo: Calibrazione | Rosso: Max Diam")
    ax[0].axis('off')

    ax[1].imshow(mask_rot, cmap='gray')
    ax[1].set_title("Maschera Analisi")
    ax[1].axis('off')

    viz_res = cv2.cvtColor(img_rot, cv2.COLOR_GRAY2RGB)
    if len(sx_v) > 0:
        cv2.rectangle(viz_res,
                      (int(sx_v[0]), int(y_t_sub)),
                      (int(dx_v[0]), int(y_b_sub)),
                      (0, 255, 0), 2)
    ax[2].imshow(viz_res)
    ax[2].set_title(f"Vol: {volume:.2E} µm³")
    ax[2].axis('off')

    plt.suptitle(os.path.basename(file_path), fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return {
        "File":    os.path.basename(file_path),
        "Diam_um": round(d_max_px * um_per_px, 2),
        "Alt_um":  round(h_px * um_per_px, 2),
        "Vol_um3": f"{volume:.3E}",
    }


# ==============================================================================
# ESECUZIONE — analizza tutti i file supportati nella cartella corrente
# ==============================================================================
path_folder = '.'
results = []

for f in sorted(os.listdir(path_folder)):
    if f.lower().endswith(ESTENSIONI_SUPPORTATE):
        res = analizza_singola_immagine(
            os.path.join(path_folder, f),
            D_FIBRA_REALE=D_FIBRA_REALE
        )
        if res:
            results.append(res)

if results:
    df = pd.DataFrame(results)
    display(df)
else:
    print("Nessun file analizzato. Controlla la cartella e le estensioni supportate.")